# Restaurant and Dish Candidate Filtering

This notebook converts cuisine and spending predictions into real restaurant dishes. Deterministic filters run before RAG and LLM generation, so later stages receive only records that exist in the Swiggy dataset.

**Safety boundary:** Restaurant records do not provide complete ingredient or cross-contamination information. Every candidate must still pass the separate dietary and allergy safety pipeline.

## 1. Setup

### Import libraries

In [1]:
from pathlib import Path
from collections.abc import Mapping

import pandas as pd

### Locate the project and final restaurant dataset

In [2]:
def find_project_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data" / "user_order_history.csv").is_file():
            return folder
        nested_project = folder / "culinary_matchmaker"
        if (nested_project / "data" / "user_order_history.csv").is_file():
            return nested_project
    raise FileNotFoundError("Could not locate the culinary_matchmaker project")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "swiggy_cleaned_sample_expanded.csv"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Restaurant dataset not found: {DATA_PATH}")

print("Dataset:", DATA_PATH)

Dataset: D:\The Intelligent Culinary Matchmaker & Personalized Menu Agent\culinary_matchmaker\data\swiggy_cleaned_sample_expanded.csv


## 2. Load and Prepare Dishes

### Load only the columns used by filtering and ranking

In [3]:
required_columns = [
    "restaurant_id", "restaurant_name", "city", "locality", "address",
    "cuisine", "menu_category", "dish_name", "dish_price", "veg_nonveg",
    "restaurant_rating", "rating_count",
]

dishes = pd.read_csv(DATA_PATH, usecols=required_columns, low_memory=False)
print("Loaded rows:", len(dishes))

Loaded rows: 100000


### Define text normalization

Cuisine fields may contain several comma-separated labels. Delimited normalized tokens let `Chinese` match `Asian, Chinese` without unsafe partial-text matching.

In [4]:
def text_key(value):
    return " ".join(str(value).strip().casefold().split())


def cuisine_key(value):
    if pd.isna(value):
        return "|"
    labels = {text_key(label) for label in str(value).split(",")}
    labels.discard("")
    return "|" + "|".join(sorted(labels)) + "|"

### Clean fields and add internal matching keys

In [5]:
def prepare_dishes(frame):
    missing_columns = set(required_columns) - set(frame.columns)
    if missing_columns:
        raise ValueError(f"Dataset is missing columns: {sorted(missing_columns)}")

    cleaned = frame[required_columns].copy()
    cleaned["dish_price"] = pd.to_numeric(cleaned["dish_price"], errors="coerce")
    cleaned["restaurant_rating"] = pd.to_numeric(
        cleaned["restaurant_rating"], errors="coerce"
    )
    cleaned["rating_count"] = pd.to_numeric(
        cleaned["rating_count"], errors="coerce"
    ).fillna(0)

    cleaned = cleaned.dropna(
        subset=["restaurant_id", "restaurant_name", "city", "dish_name", "dish_price"]
    )
    cleaned = cleaned[cleaned["dish_price"] >= 0].copy()

    cleaned["_city_key"] = cleaned["city"].map(text_key)
    cleaned["_locality_key"] = cleaned["locality"].map(text_key)
    cleaned["_dietary_key"] = cleaned["veg_nonveg"].map(text_key)
    cleaned["_cuisine_key"] = cleaned["cuisine"].map(cuisine_key)
    cleaned["_dish_key"] = cleaned["dish_name"].map(text_key)

    cleaned = cleaned.drop_duplicates(
        subset=["restaurant_id", "_dish_key"], keep="first"
    )
    return cleaned.reset_index(drop=True)

### Prepare the complete restaurant dataset

In [6]:
dishes_clean = prepare_dishes(dishes)
print("Clean unique restaurant-dish rows:", len(dishes_clean))

Clean unique restaurant-dish rows: 100000


## 3. Normalize User and Model Inputs

### Choose the final budget

In [7]:
def positive_amount(value, name):
    if isinstance(value, bool):
        raise ValueError(f"{name} must be a positive number")
    try:
        amount = float(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be a positive number") from error
    if not pd.notna(amount) or amount <= 0:
        raise ValueError(f"{name} must be a positive number")
    return amount


### Normalize the dietary preference

In [8]:
def dietary_key(preference):
    if preference is None or text_key(preference) in {"", "any", "no preference"}:
        return None
    key = text_key(preference).replace("_", "-")
    aliases = {
        "veg": "veg", "vegetarian": "veg",
        "non-veg": "non-veg", "non veg": "non-veg",
        "nonvegetarian": "non-veg", "non-vegetarian": "non-veg",
    }
    if key == "vegan":
        raise ValueError(
            "The restaurant dataset cannot verify vegan compatibility; "
            "use the ingredient and safety layers."
        )
    if key not in aliases:
        raise ValueError("dietary_preference must be Veg, Non-veg, or None")
    return aliases[key]

### Preserve the classifier's ranked cuisine predictions

In [9]:
def ranked_cuisines(predictions):
    items = [predictions] if isinstance(predictions, str) else predictions
    cuisines, seen = [], set()

    for item in items:
        cuisine = item.get("cuisine") if isinstance(item, Mapping) else item
        if cuisine is None or not text_key(cuisine):
            continue

        label = str(cuisine).strip()
        key = text_key(label)
        if key not in seen:
            seen.add(key)
            cuisines.append(label)

    if not cuisines:
        raise ValueError("At least one predicted cuisine is required")
    return cuisines

## 4. Filter and Rank Candidates

### Rank higher-rated and better-supported restaurants first

In [10]:
def rank_candidates(frame, top_n):
    ranked = frame.sort_values(
        by=["restaurant_rating", "rating_count", "dish_price"],
        ascending=[False, False, True],
        na_position="last",
        kind="stable",
    ).head(top_n)

    result = ranked[required_columns].reset_index(drop=True).copy()
    result.insert(0, "rank", range(1, len(result) + 1))
    return result

### Apply hard constraints

City, budget, dietary preference, and minimum rating are strict requirements. Fallback logic never relaxes them.

In [11]:
def apply_hard_constraints(frame, city, budget, preference, minimum_rating):
    candidates = frame[
        (frame["_city_key"] == text_key(city))
        & (frame["dish_price"] <= budget)
    ]
    if preference is not None:
        candidates = candidates[candidates["_dietary_key"] == preference]
    if minimum_rating is not None:
        candidates = candidates[candidates["restaurant_rating"] >= minimum_rating]
    return candidates

### Try exact and fallback matches

The search tries the top cuisine in the exact locality, then citywide, before repeating those scopes for lower-ranked cuisines. Citywide fallback is not distance search because this CSV has no coordinates.

### Search locality and cuisine fallback stages

The top cuisine is checked in the exact locality and then citywide. Lower-ranked cuisines follow the same order. This helper returns the first successful stage without relaxing hard constraints.

In [12]:
def find_candidate_match(base_candidates, cuisines, locality, allow_citywide_fallback):
    scopes = [("exact_locality", False)] if locality else [("citywide", False)]
    if locality and allow_citywide_fallback:
        scopes.append(("citywide", True))

    for cuisine_index, cuisine in enumerate(cuisines):
        cuisine_token = f"|{text_key(cuisine)}|"
        cuisine_candidates = base_candidates[
            base_candidates["_cuisine_key"].str.contains(cuisine_token, regex=False)
        ]

        for scope, locality_relaxed in scopes:
            candidates = cuisine_candidates
            if scope == "exact_locality":
                candidates = candidates[candidates["_locality_key"] == locality]
            if candidates.empty:
                continue

            relaxed_constraints = []
            if locality_relaxed:
                relaxed_constraints.append("locality")
            if cuisine_index > 0:
                relaxed_constraints.append("predicted_cuisine")

            match_stage = scope if cuisine_index == 0 else f"alternate_cuisine_{scope}"
            return candidates, cuisine, match_stage, tuple(relaxed_constraints)

    return base_candidates.iloc[0:0], None, "no_match", ()

### Combine validation, hard filtering, fallback, and ranking

In [13]:
def filter_dish_candidates(
    frame,
    *,
    user_city,
    user_locality,
    predicted_cuisines,
    user_budget,
    dietary_preference=None,
    minimum_rating=None,
    top_n=10,
    allow_citywide_fallback=True,
):
    if not text_key(user_city):
        raise ValueError("user_city is required")
    if isinstance(top_n, bool) or not isinstance(top_n, int) or not 1 <= top_n <= 100:
        raise ValueError("top_n must be an integer between 1 and 100")
    if minimum_rating is not None:
        minimum_rating = float(minimum_rating)
        if not 0 <= minimum_rating <= 5:
            raise ValueError("minimum_rating must be between 0 and 5")

    final_budget = positive_amount(user_budget, "user_budget")
    cuisines = ranked_cuisines(predicted_cuisines)
    preference = dietary_key(dietary_preference)
    locality = text_key(user_locality) if user_locality else ""

    base_candidates = apply_hard_constraints(
        frame, user_city, final_budget, preference, minimum_rating
    )
    matched_rows, selected_cuisine, match_stage, relaxed_constraints = (
        find_candidate_match(
            base_candidates, cuisines, locality, allow_citywide_fallback
        )
    )

    return {
        "candidates": rank_candidates(matched_rows, top_n),
        "final_budget": final_budget,
        "selected_cuisine": selected_cuisine,
        "match_stage": match_stage,
        "relaxed_constraints": relaxed_constraints,
        "requires_safety_screening": True,
    }

## 5. Example Recommendation Candidates

### Run the filter with classifier and user inputs

In [14]:
example = filter_dish_candidates(
    dishes_clean,
    user_city="Bengaluru",
    user_locality="HSR",
    predicted_cuisines=[
        {"cuisine": "North Indian", "probability": 0.78},
        {"cuisine": "Chinese", "probability": 0.16},
    ],
    dietary_preference="Vegetarian",
    user_budget=250,
    minimum_rating=4.0,
    top_n=10,
)

print("Selected cuisine:", example["selected_cuisine"])
print("Budget:", example["final_budget"])
print("Match stage:", example["match_stage"])
print("Relaxed constraints:", example["relaxed_constraints"])
print("Safety screening required:", example["requires_safety_screening"])
example["candidates"]

Selected cuisine: North Indian
Budget: 250.0
Match stage: exact_locality
Relaxed constraints: ()
Safety screening required: True


,rank,restaurant_id,restaurant_name,city,locality,address,cuisine,menu_category,dish_name,dish_price,veg_nonveg,restaurant_rating,rating_count
0,1,171768,Mumbai Tiffin,Bengaluru,HSR,"Mumbai Tiffin, 2345, 17th Cross, Opposite Wate...","North Indian, Home Food",Individual Items,Phulka,23,Veg,4.6,1000
1,2,171768,Mumbai Tiffin,Bengaluru,HSR,"Mumbai Tiffin, 2345, 17th Cross, Opposite Wate...","North Indian, Home Food",Sides,Onion And Lemon Salad,28,Veg,4.6,1000
2,3,171768,Mumbai Tiffin,Bengaluru,HSR,"Mumbai Tiffin, 2345, 17th Cross, Opposite Wate...","North Indian, Home Food",Stuffed Paratha,Gobi Paratha,166,Veg,4.6,1000
3,4,539039,The Rush Restaurant,Bengaluru,HSR,"The Rush Restaurant, 'Ground Floor No 1160 5th...","North Indian, Chinese",Tandoori Breads,Plain Parantha,60,Veg,4.6,100
4,5,539039,The Rush Restaurant,Bengaluru,HSR,"The Rush Restaurant, 'Ground Floor No 1160 5th...","North Indian, Chinese",Momos,Veg Fried Momo,99,Veg,4.6,100
5,6,539039,The Rush Restaurant,Bengaluru,HSR,"The Rush Restaurant, 'Ground Floor No 1160 5th...","North Indian, Chinese",Tandoori Breads,Stuffed Naan,105,Veg,4.6,100
6,7,539039,The Rush Restaurant,Bengaluru,HSR,"The Rush Restaurant, 'Ground Floor No 1160 5th...","North Indian, Chinese",Momos,Tandoori Paneer Momo,119,Veg,4.6,100
7,8,539039,The Rush Restaurant,Bengaluru,HSR,"The Rush Restaurant, 'Ground Floor No 1160 5th...","North Indian, Chinese",Tandoori Breads,Mooli Parantha,130,Veg,4.6,100
8,9,539039,The Rush Restaurant,Bengaluru,HSR,"The Rush Restaurant, 'Ground Floor No 1160 5th...","North Indian, Chinese",Sandwiches,Veg Cheese Sandwich,149,Veg,4.6,100
9,10,539039,The Rush Restaurant,Bengaluru,HSR,"The Rush Restaurant, 'Ground Floor No 1160 5th...","North Indian, Chinese",Main Course,Paneer Do Pyaza,199,Veg,4.6,100


## Workflow Boundary

The returned rows are grounded restaurant candidates, not safety-approved recommendations. The next stage must retrieve ingredient knowledge and apply deterministic allergy and dietary safety rules before an LLM explains any result.